In [1]:
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader
from sklearn.model_selection import StratifiedKFold
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from optional_fine_tune import BertForTweetClassification, DfToDataset, train_epoch, eval_model

D:\Codding\Education\NLP\Natural Language Processing with Disaster Tweets\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train = pd.read_csv("data/train.csv")
train['keyword'] = train['keyword'].fillna("")
train['combined_text'] = "Keyword: " + train['keyword'] + ". Text: " + train['text']

In [3]:
texts = train['combined_text'].values
targets = train['target'].values

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используем устройство: {device}")

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
train_dataset = DfToDataset(texts, targets, tokenizer)

Используем устройство: cuda


In [5]:
k_fold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_f1_scores = []

for fold, (train_idx, val_idx) in enumerate(k_fold.split(texts, targets)):
    X_train, y_train = texts[train_idx], targets[train_idx]
    X_val, y_val = texts[val_idx], targets[val_idx]

    train_dataset = DfToDataset(X_train, y_train, tokenizer)
    val_dataset = DfToDataset(X_val, y_val, tokenizer)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

    model = BertForTweetClassification(MODEL_NAME).to(device)

    EPOCHS = 3
    loss_fn = nn.CrossEntropyLoss().to(device)
    optimizer = AdamW(model.parameters(), lr=2e-5)

    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps
    )

    best_fold_f1 = 0
    for epoch in range(EPOCHS):
        train_loss = train_epoch(model, optimizer, train_loader, loss_fn, scheduler, device)
        val_f1 = eval_model(model, val_loader, device)

        print(f"Эпоха {epoch + 1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val F1: {val_f1:.4f}")

        # Сохраняем лучший скор внутри фолда
        if val_f1 > best_fold_f1:
            best_fold_f1 = val_f1

            model_path = f"models/best_bert_fold_{fold + 1}.pt"
            torch.save(model.state_dict(), model_path)
            print(f"Веса модели сохранены в {model_path}")

    print(f"Лучший F1 для фолда {fold + 1}: {best_fold_f1:.4f}")
    cv_f1_scores.append(best_fold_f1)


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 11713.97it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Эпоха 1/3 | Train Loss: 0.4718 | Val F1: 0.8038
 => Веса модели сохранены в best_bert_fold_1.pt
Эпоха 2/3 | Train Loss: 0.3461 | Val F1: 0.8119
 => Веса модели сохранены в best_bert_fold_1.pt
Эпоха 3/3 | Train Loss: 0.2874 | Val F1: 0.8122
 => Веса модели сохранены в best_bert_fold_1.pt
-> Лучший F1 для фолда 1: 0.8122


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 15377.27it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Эпоха 1/3 | Train Loss: 0.4984 | Val F1: 0.7782
 => Веса модели сохранены в best_bert_fold_2.pt
Эпоха 2/3 | Train Loss: 0.3510 | Val F1: 0.7937
 => Веса модели сохранены в best_bert_fold_2.pt
Эпоха 3/3 | Train Loss: 0.2884 | Val F1: 0.7955
 => Веса модели сохранены в best_bert_fold_2.pt
-> Лучший F1 для фолда 2: 0.7955


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 11099.57it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Эпоха 1/3 | Train Loss: 0.4787 | Val F1: 0.7914
 => Веса модели сохранены в best_bert_fold_3.pt
Эпоха 2/3 | Train Loss: 0.3435 | Val F1: 0.7944
 => Веса модели сохранены в best_bert_fold_3.pt
Эпоха 3/3 | Train Loss: 0.2840 | Val F1: 0.8044
 => Веса модели сохранены в best_bert_fold_3.pt
-> Лучший F1 для фолда 3: 0.8044


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 12471.54it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Эпоха 1/3 | Train Loss: 0.4756 | Val F1: 0.7797
 => Веса модели сохранены в best_bert_fold_4.pt
Эпоха 2/3 | Train Loss: 0.3509 | Val F1: 0.8058
 => Веса модели сохранены в best_bert_fold_4.pt
Эпоха 3/3 | Train Loss: 0.2908 | Val F1: 0.8097
 => Веса модели сохранены в best_bert_fold_4.pt
-> Лучший F1 для фолда 4: 0.8097


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 11673.22it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Эпоха 1/3 | Train Loss: 0.4823 | Val F1: 0.7946
 => Веса модели сохранены в best_bert_fold_5.pt
Эпоха 2/3 | Train Loss: 0.3494 | Val F1: 0.8000
 => Веса модели сохранены в best_bert_fold_5.pt
Эпоха 3/3 | Train Loss: 0.2799 | Val F1: 0.8053
 => Веса модели сохранены в best_bert_fold_5.pt
-> Лучший F1 для фолда 5: 0.8053


In [6]:
print(f"ИТОГОВЫЙ СРЕДНИЙ FINE-TUNED F1-SCORE: {np.mean(cv_f1_scores):.4f}")


ИТОГОВЫЙ СРЕДНИЙ FINE-TUNED F1-SCORE: 0.8054
